In [12]:
import urllib.request, os
import json
import ssl

# This restores the same behavior as before.
context = ssl._create_unverified_context()

# Variables

list_layernames = ['pl_20092','pl_20093','pl_20094','pl_2010','pl_20121','pl_20122','pl_2012','pl_20141',
                  'pl_20142','pl_2014','pl_2500','pl_3000','pl_50011','pl_5001']

def loop_all_layers(input_layer):
    
    layerName = input_layer
    nameListFID = layerName+ ".json"
    FolderJsons = "Output_"+ layerName
    ID = "objectid" #FID or OBJECTID

    myUrl = "https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana"
    myService = "/"+layerName+"/MapServer/0/"

    multiplier1st = 1000
    print("initiating multiplier " + str(multiplier1st))


    myParams = "query?where="+ID+">-2&text=&objectIds=&time=&geometry=&geometryType=esriGeometryEnvelope&inSR=&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=false&returnTrueCurves=false&maxAllowableOffset=&geometryPrecision=&outSR=&returnIdsOnly=true&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&outStatistics=&returnZ=false&returnM=false&gdbVersion=&returnDistinctValues=false&resultOffset=&resultRecordCount=&f=pjson"

    # Query ArcGIS Server Map Service
    myRequest = myUrl + myService + myParams
    #r = http.request('GET', myRequest)
    response = urllib.request.urlopen(myRequest,context=context)
    #myJSON = json.loads(r.data.decode('utf-8'))
    myJSON = response.read()

    # Write response to json text file

    foo = open(nameListFID, "wb")
    foo.write(myJSON);
    foo.close()
    '''
    with open(nameListFID, 'w') as json_file:
      json.dump(myJSON, json_file)
    '''
    current_directory = os.getcwd()
    final_directory = os.path.join(current_directory,FolderJsons)
    if not os.path.exists(final_directory):
        os.makedirs(final_directory)

    json_data = open(nameListFID, "r").read()
    data = json.loads(json_data)
    listD = data['objectIds']
    lenD = max(listD)
    print("Calculating maximum object id: " + str(lenD))

    divMult1 = int(lenD / multiplier1st) #1000
    modMult1 = int(lenD % multiplier1st)
    if modMult1 != 0:
        divMult1 += 1

    def multiplier2nd(maxb, divider):
        a = []
        multiplier = int(multiplier1st/divider)
        divMult2 = int(maxb / multiplier)
        modMult2 = int(maxb % multiplier)
        if modMult2 != 0:
            divMult2 += 1
        a.append(multiplier)
        a.append(divMult2)
        a.append(modMult2)
        return a

    def readWebgis(a,b):
        print ("Processing FID: [" + a + ","+ b + "]")
        ParamText1 = "query?where="+ID+"+>%3D+"          #CHANGE
        ParamTextAnd = "+AND+"+ID+"+<"
        ParamText3 = "&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson"
        iterRequest = myUrl + myService + ParamText1 + a + ParamTextAnd + b + ParamText3
        print("reading URL WEBGIS: " + iterRequest)
        response = urllib.request.urlopen(iterRequest,context=context)
        #r = http.request('GET', myRequest)
        print("URL opened!")
        print ("Reading JSON files: [" + a + ","+ b + "]")
        #myJSON = json.loads(r.data.decode('utf-8'))
        myJSON = response.read()
        dirF = os.path.join(final_directory,"jsonOutput"+b+".json")
        # Write response to json text file
        foo = open(dirF, "wb")
        foo.write(myJSON);
        foo.close()
        '''
        with open(nameListFID, 'w') as json_file:
            json.dump(myJSON, json_file)
        '''
        global fileSize
        fileSize = os.stat(dirF).st_size
        print ("checking file size [" + a + ","+ b + "], (error 500 indicator: if Filesize < 1kb)")
        return dirF
    listErrA = []
    listErrB = []

    for i in range(divMult1): #1000
        #if i < 98:
         #   continue
        a = str(i*multiplier1st)
        if i*multiplier1st + multiplier1st <= lenD:
            b = str(i*multiplier1st + multiplier1st)
        else:
            b = str(i*multiplier1st + modMult1)
        readWebgis(a,b)
        if fileSize < 100:
            listErrA.append(int(a))
            listErrB.append(int(b))
            print ("Error found in : [" + a + ","+ b + "] \n--------appending to list A and B")
        #if i < lenD:
        #    break
    print(listErrA)
    print(listErrB)
    listErr2A = []      #500 * 2 = 1000
    listErr2B = []
    def Loopmultiplier(listErrA, listErrB, divider, listErr2A,listErr2B):
        for i in range(len(listErrA)):
            a = listErrA[i]
            maxb = listErrB[i]
            multiplier2nd(maxb, divider)
            multiplier = multiplier2nd(maxb,divider)[0]
            divMult = multiplier2nd(maxb,divider)[1]
            modMult = multiplier2nd(maxb,divider)[2]
            print("trying multiplier :" +str(multiplier) )
            for i in range(divMult):
                if i*multiplier < a:
                    continue
                A = str(i*multiplier)
                if i*multiplier+multiplier <= maxb:
                    B = str(i*multiplier + multiplier)
                else:
                    B = str(i*multiplier + modMult)
                readWebgis(A,B)
                if fileSize < 100:
                    listErr2A.append(int(A))
                    listErr2B.append(int(B))
                    print ("Error found in : [" + A + ","+ B + "] \n--------appending to list A and B")
            print ("Processed JSON file " + str(i+1) + " of " + str(divMult) + "in multiplier : "+ str(multiplier) + "\n---------------")
        print(listErr2A)
        print(listErr2B)

    '''
    # Create Feature Class
    ws = os.getcwd() + os.sep
    arcpy.JSONToFeatures_conversion("jsonOutput.json", ws + "finalShapfile.shp")
    '''
    Loopmultiplier(listErrA, listErrB, 2, listErr2A,listErr2B)
    listErr3A = []      #250 * 4 = 1000
    listErr3B = []
    Loopmultiplier(listErr2A, listErr2B, 4, listErr3A,listErr3B)
    listErr4A = []      #125 * 8 = 1000
    listErr4B = []
    Loopmultiplier(listErr3A, listErr3B, 8, listErr4A,listErr4B)
    listErr5A = []      #25 * 40 = 1000
    listErr5B = []
    Loopmultiplier(listErr4A, listErr4B, 40, listErr5A,listErr5B)
    listErr6A = []      #5 * 200 = 1000
    listErr6B = []
    Loopmultiplier(listErr5A, listErr5B, 200, listErr6A,listErr6B)
    listErr7A = []      #1 * 1000 = 1000
    listErr7B = []
    Loopmultiplier(listErr6A, listErr6B, 200, listErr7A,listErr7B)

In [13]:
for i in list_layernames:
    loop_all_layers(i)

initiating multiplier 1000
Calculating maximum object id: 2446556
Processing FID: [0,1000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+0+AND+objectid+<1000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson
URL opened!
Reading JSON files: [0,1000]
checking file size [0,1000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [1000,2000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+1000+AND+objectid+<2000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsO

URL opened!
Reading JSON files: [13000,14000]
checking file size [13000,14000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [14000,15000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+14000+AND+objectid+<15000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson
URL opened!
Reading JSON files: [14000,15000]
checking file size [14000,15000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [15000,16000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+15000+AND+objectid+<16000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&rela

URL opened!
Reading JSON files: [27000,28000]
checking file size [27000,28000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [28000,29000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+28000+AND+objectid+<29000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson
URL opened!
Reading JSON files: [28000,29000]
checking file size [28000,29000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [29000,30000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+29000+AND+objectid+<30000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&rela

URL opened!
Reading JSON files: [41000,42000]
checking file size [41000,42000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [42000,43000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+42000+AND+objectid+<43000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson
URL opened!
Reading JSON files: [42000,43000]
checking file size [42000,43000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [43000,44000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+43000+AND+objectid+<44000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&rela

URL opened!
Reading JSON files: [55000,56000]
checking file size [55000,56000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [56000,57000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+56000+AND+objectid+<57000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson
URL opened!
Reading JSON files: [56000,57000]
checking file size [56000,57000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [57000,58000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+57000+AND+objectid+<58000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&rela

...

URL opened!
Reading JSON files: [934000,935000]
checking file size [934000,935000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [935000,936000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+935000+AND+objectid+<936000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson
URL opened!
Reading JSON files: [935000,936000]
checking file size [935000,936000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [936000,937000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+936000+AND+objectid+<937000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRe

checking file size [947000,948000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [948000,949000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+948000+AND+objectid+<949000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson
URL opened!
Reading JSON files: [948000,949000]
checking file size [948000,949000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [949000,950000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+949000+AND+objectid+<950000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeo

URL opened!
Reading JSON files: [961000,962000]
checking file size [961000,962000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [962000,963000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+962000+AND+objectid+<963000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson
URL opened!
Reading JSON files: [962000,963000]
checking file size [962000,963000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [963000,964000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+963000+AND+objectid+<964000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRe

checking file size [974000,975000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [975000,976000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+975000+AND+objectid+<976000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeometry=true&geometryPrecision=&outSR=&returnIdsOnly=false&returnCountOnly=false&orderByFields=&groupByFieldsForStatistics=&returnZ=false&returnM=false&returnDistinctValues=false&returnTrueCurves=false&f=pjson
URL opened!
Reading JSON files: [975000,976000]
checking file size [975000,976000], (error 500 indicator: if Filesize < 1kb)
Processing FID: [976000,977000]
reading URL WEBGIS: https://nfms.menlhk.go.id:8443/arcgis/rest/services/simontana/pl_20092/MapServer/0/query?where=objectid+>%3D+976000+AND+objectid+<977000&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelIntersects&relationParam=&outFields=*&returnGeo

RemoteDisconnected: Remote end closed connection without response